In [ ]:
# -*- coding: utf-8 -*-
"""
Arquivo: treinar.py
Treina os modelos de Conversão e Fator e salva em disco.
"""

import joblib
from pathlib import Path
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

# -------------------
# Função para criar features simples
# -------------------
def build_features(df):
    df = df.copy()
    df["texto"] = (df["MATERIAL"].astype(str) + " " + df["NOME_CONC"].astype(str)).fillna("")
    return df[["texto"]]

# -------------------
# Função principal
# -------------------
def treinar_modelos(caminho_treino):
    df = pd.read_excel(caminho_treino)

    # Ajusta colunas
    df["MATERIAL"] = df["MATERIAL"].astype(str).fillna("desconhecido")
    df["NOME_CONC"] = df["NOME_CONC"].astype(str).fillna("desconhecido")
    df["CONVERSAO"] = df["CONVERSAO"].astype(str).fillna("desconhecido")
    df["FATOR"] = pd.to_numeric(df["FATOR"], errors="coerce").fillna(0.0)

    # Features
    X = build_features(df)
    y_conv = df["CONVERSAO"]
    y_fator = df["FATOR"]

    # Vetorizador de texto
    tfidf = TfidfVectorizer(ngram_range=(1, 2), max_features=5000)

    preprocessor = ColumnTransformer([
        ("txt", tfidf, "texto")
    ], remainder="drop")

    # Classificador para CONVERSAO
    modelo_conv = Pipeline([
        ("pre", preprocessor),
        ("clf", RandomForestClassifier(n_estimators=150, n_jobs=-1, random_state=42))
    ])

    # Regressor para FATOR
    modelo_fator = Pipeline([
        ("pre", preprocessor),
        ("reg", RandomForestRegressor(n_estimators=150, n_jobs=-1, random_state=42))
    ])

    print("Treinando modelo de CONVERSAO...")
    modelo_conv.fit(X, y_conv)

    print("Treinando modelo de FATOR...")
    modelo_fator.fit(X, y_fator)

    # Salva modelos
    joblib.dump(modelo_conv, "modelo_conversao.pkl")
    joblib.dump(modelo_fator, "modelo_fator.pkl")
    print("Modelos salvos em disco: modelo_conversao.pkl e modelo_fator.pkl")

    return modelo_conv, modelo_fator


if __name__ == "__main__":
    caminho_treino = r"C:\Users\SeuUsuario\Downloads\Banco_de_Fardos.xlsx"  # <<< ajuste
    treinar_modelos(caminho_treino)


In [ ]:
# -*- coding: utf-8 -*-
"""
Arquivo: prever.py
Carrega modelos já treinados e aplica em novos fardos.
"""

import joblib
import pandas as pd
import numpy as np

# -------------------
# Função para criar features
# -------------------
def build_features(df):
    df = df.copy()
    df["texto"] = (df["MATERIAL"].astype(str) + " " + df["NOME_CONC"].astype(str)).fillna("")
    return df[["texto"]]

# -------------------
# Função para arredondar fatores
# -------------------
def arredondar_fator(valor):
    if valor is None:
        return 0
    # força positivo
    valor = float(valor)

    # se está bem perto de um inteiro → arredonda
    if abs(valor - round(valor)) < 0.05:  
        return int(round(valor))

    # caso contrário, arredonda para no máximo 2 casas decimais
    return round(valor, 2)

# -------------------
# Função principal
# -------------------
def prever(modelo_conv, modelo_fator, caminho_novos, caminho_saida=None):
    df = pd.read_excel(caminho_novos)

    # Ajusta colunas
    df["MATERIAL"] = df["MATERIAL"].astype(str).fillna("desconhecido")
    df["NOME_CONC"] = df["NOME_CONC"].astype(str).fillna("desconhecido")

    # Features
    X_new = build_features(df)

    # Predições
    df["CONVERSAO_PREV"] = modelo_conv.predict(X_new)
    fatores_prev = modelo_fator.predict(X_new)

    # Aplica arredondamento inteligente
    df["FATOR_PREV"] = [arredondar_fator(f) for f in fatores_prev]

    # Salvar resultado
    if caminho_saida:
        df.to_excel(caminho_saida, index=False)
        print(f"Resultado salvo em: {caminho_saida}")

    return df


if __name__ == "__main__":
    # Carregar modelos treinados
    modelo_conv = joblib.load("modelo_conversao.pkl")
    modelo_fator = joblib.load("modelo_fator.pkl")

    caminho_novos = r"C:\Users\SeuUsuario\Downloads\novos_fardos.xlsx"   # <<< ajuste
    saida = r"C:\Users\SeuUsuario\Downloads\Resultado_Prev.xlsx"        # <<< ajuste

    resultado = prever(modelo_conv, modelo_fator, caminho_novos, caminho_saida=saida)
    print(resultado.head(20))
